<a href="https://colab.research.google.com/github/alikhrzh/ML-project/blob/main/club_rec_categorize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('updated_clubs.csv')

In [10]:
categories = df['category'].unique().tolist()

In [22]:
import json

with open("student_club_recommendation_examples.json", "r", encoding="utf-8-sig") as f:
    data = json.load(f)

print(len(data))
print(data[0])

200
{'goal': 'I want to prepare for math competitions and sharpen my skills', 'interest': 'mathematics, puzzles, competitions', 'text': "I'm still figuring out campus life, but I know I'd enjoy a club where I can sharpen my math skills. Something centered on math problems and competitions sounds right.", 'categories': ['academic']}


In [23]:
X = []
for sample in data:
    combined_text = f"Goal: {sample['goal']} Interest: {sample['interest']} Text: {sample['text']}"
    X.append(combined_text)

print(X[0])

Goal: I want to prepare for math competitions and sharpen my skills Interest: mathematics, puzzles, competitions Text: I'm still figuring out campus life, but I know I'd enjoy a club where I can sharpen my math skills. Something centered on math problems and competitions sounds right.


In [26]:
label_list = ['cultural', 'academic', 'sports', 'professional', 'social', 'art']
y = []
for sample in data:
    label_vector = [1 if label in sample["categories"] else 0 for label in label_list]
    y.append(label_vector)
y = np.array(y)

print(y[0])

[0 1 0 0 0 0]


In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(max_iter=2000)
    ))
])

In [29]:
model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('clf',
                 OneVsRestClassifier(estimator=LogisticRegression(max_iter=2000)))])

In [31]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

In [36]:
from sklearn.metrics import f1_score

print("F1: ", f1_score(y_test, y_pred, average='weighted'))

F1:  0.599777867889323


In [56]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import SGDClassifier

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )),
    ("clf", OneVsRestClassifier(
        SGDClassifier(loss="hinge", random_state=42)
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
# y_prob = model.predict_proba(X_test)
# y_pred = (y_prob >= 0.3).astype(int)

print("F1: ", f1_score(y_test, y_pred, average='weighted'))

F1:  0.9268359268359269


In [58]:
examples = [
    "Goal: improve coding skills Interest: ML, hackathons Text: I want to join a club where I can participate in ML hackathons.",
    "Goal: stay active Interest: football, fitness Text: I want a club where I can train and play football.",
    "Goal: meet people Interest: community, events Text: I want to make new friends and join fun group events."
]

preds = model.predict(examples)
# y_prob = model.predict_proba(examples)

# threshold = 0.3
# preds = (y_prob >= threshold).astype(int)

for text, probs, pred in zip(examples, y_prob, preds):
    print("TEXT:", text)

    ranked = sorted(zip(label_list, probs), key=lambda x: x[1], reverse=True)
    print("Scores:")
    for label, score in ranked:
        print(f"  {label}: {score:.3f}")

    labels = [label_list[i] for i in range(len(label_list)) if pred[i] == 1]
    print("Predicted:", labels)
    print("-" * 50)

TEXT: Goal: improve coding skills Interest: ML, hackathons Text: I want to join a club where I can participate in ML hackathons.
Scores:
  art: 0.341
  academic: 0.319
  sports: 0.275
  professional: 0.256
  social: 0.226
  cultural: 0.214
Predicted: ['academic']
--------------------------------------------------
TEXT: Goal: stay active Interest: football, fitness Text: I want a club where I can train and play football.
Scores:
  sports: 0.558
  art: 0.294
  professional: 0.215
  cultural: 0.209
  social: 0.205
  academic: 0.204
Predicted: ['sports']
--------------------------------------------------
TEXT: Goal: meet people Interest: community, events Text: I want to make new friends and join fun group events.
Scores:
  social: 0.655
  art: 0.276
  professional: 0.271
  sports: 0.227
  cultural: 0.198
  academic: 0.194
Predicted: ['social']
--------------------------------------------------


In [ ]:
# trash

In [ ]:
# !pip install transformers

In [16]:
# from transformers import pipeline

# classifier = pipeline(
#     "zero-shot-classification",
#     model="facebook/bart-large-mnli"
# )

# labels = [
#     "academic club focused on learning, studying, coding, research, olympiads, or competitions",
#     "professional club focused on career growth, networking, leadership, entrepreneurship, or industry",
#     "sports club focused on physical exercise, fitness, football, basketball, and athletic training",
#     "social club focused on making friends, community, meeting people, and having fun together",
#     "cultural club focused on languages, traditions, national identity, and cultural exchange",
#     "art club focused on music, drawing, painting, performance, dance, theater, or creativity"
# ]

# result = classifier(
#     "I like coding and want to join hackathons",
#     candidate_labels=labels,
#     multi_label=True
# )

# print(result)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'I like coding and want to join hackathons', 'labels': ['academic club focused on learning, studying, coding, research, olympiads, or competitions', 'professional club focused on career growth, networking, leadership, entrepreneurship, or industry', 'social club focused on making friends, community, meeting people, and having fun together', 'sports club focused on physical exercise, fitness, football, basketball, and athletic training', 'cultural club focused on languages, traditions, national identity, and cultural exchange', 'art club focused on music, drawing, painting, performance, dance, theater, or creativity'], 'scores': [0.6860582828521729, 0.6744956970214844, 0.02202487550675869, 0.013072394765913486, 0.00032435456523671746, 0.00023617269471287727]}


In [18]:
# result = classifier(
#     "I want to join in a club where I will participate in ML hackathons",
#     candidate_labels=labels,
#     multi_label=True
# )

In [19]:
# result

{'sequence': 'I want to join in a club where I will participate in ML hackathons',
 'labels': ['sports club focused on physical exercise, fitness, football, basketball, and athletic training',
  'professional club focused on career growth, networking, leadership, entrepreneurship, or industry',
  'academic club focused on learning, studying, coding, research, olympiads, or competitions',
  'social club focused on making friends, community, meeting people, and having fun together',
  'cultural club focused on languages, traditions, national identity, and cultural exchange',
  'art club focused on music, drawing, painting, performance, dance, theater, or creativity'],
 'scores': [0.5311703681945801,
  0.44567084312438965,
  0.06944788992404938,
  0.04048515111207962,
  0.002656609285622835,
  0.0007644815486855805]}